# 3-Agent Melody Harmonization Demo

This notebook demonstrates a **3-agent pipeline** for harmonizing Bach chorale melodies
using [Microsoft Agent Framework](https://learn.microsoft.com/en-us/agent-framework/).

## Pipeline Overview

```
Bach BWV corpus (music21)
         ↓
   Soprano melody → ABC notation
         ↓
┌─────────────────────────────────────┐
│         Microsoft Agent Framework   │
│                                     │
│  [Theory Agent - Claude Sonnet 4.6] │
│   Analyzes melody → Roman numerals  │
│                  ↓                  │
│  [Harmonizer Agent - GPT-4o]        │
│   Generates chord progression V:2   │
│                  ↓                  │
│  [Orchestrator Agent - GPT-4o-mini] │
│   Validates and cleans ABC output   │
└─────────────────────────────────────┘
         ↓
   2-voice ABC → MIDI → Audio (FluidSynth)
```

## Architecture Notes
- **Workflow** (not group-chat): the two processing agents are connected as a
  directed graph via `WorkflowBuilder`. Execution is sequential and explicit.
- **Typed messages**: `MelodyInput → TheoryAnalysis → HarmonizationResult`
  keep each step's I/O visible and debuggable.
- **Configurable models**: each agent can be swapped to any supported provider
  (Anthropic, OpenAI, Ollama, Azure) without changing the pipeline code.
- **OMT in-context learning**: agent system prompts embed Open Music Theory
  chapters for harmonic functions, cadences, and phrase syntax.

## 1. Setup

Install dependencies (run once):
```bash
pip install agent-framework agent-framework-openai agent-framework-anthropic --pre
pip install python-dotenv
```

Set your API keys in `.env` at the project root:
```
OPENAI_API_KEY=sk-...
ANTHROPIC_API_KEY=sk-ant-...
```

In [ ]:
import sys
import pathlib
import asyncio
import tempfile
import os

from dotenv import load_dotenv
from IPython.display import display, HTML

# Add project root to path so we can import util and basic_agent_framework
project_root = pathlib.Path.cwd().parent
sys.path.insert(0, str(project_root))

# Load .env from project root
load_dotenv(project_root / ".env")

# Verify API keys are set
for key in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY"]:
    assert os.getenv(key), f"{key} not set — add it to .env at the project root"

# Load our framework modules
from basic_agent_framework import (
    harmonize_melody,
    load_bach_melody,
    build_harmonization_template,
    clean_abc_for_llm,
    AVAILABLE_BWV,
)
from util import abc_sonify as abc

# SoundFont path for audio synthesis
SF2_PATH = str(project_root / "data" / "soundfonts" / "GeneralUser_GS.sf2")

print("Setup complete!")
print(f"Available Bach chorales: {AVAILABLE_BWV}")
print(f"SoundFont: {SF2_PATH}")

## 2. Load a Bach Melody

We extract the **soprano voice** (melody) from BWV 253 using music21's built-in
Bach corpus, then build a 2-voice template:
- **V:1** — the original soprano melody
- **V:2** — placeholder rests (the LLM will fill these in)

In [ ]:
BWV = "bwv253"   # Change this to try other chorales: bwv255, bwv269, bwv274

# Step 1: Load the soprano melody from music21 corpus
single_voice_abc = load_bach_melody(BWV, measures=(1, 8))

# Step 2: Build the 2-voice template (V:1 = melody, V:2 = rests)
melody_template = build_harmonization_template(
    single_voice_abc,
    title_override=f"{BWV.upper()} Soprano (Bach)"
)

# Step 3: Clean the template for LLM consumption
# (removes lyrics, fermata markers, and linebreak directives)
melody_clean = clean_abc_for_llm(melody_template)

print("=== 2-Voice ABC Template (cleaned for LLM) ===")
print(melody_clean)

### Listen to the raw melody
Before running the agents, let's hear what the soprano melody sounds like by itself.

In [ ]:
# Write the melody template to a temp file and load it with util
with tempfile.NamedTemporaryFile(mode="w", suffix=".abc", delete=False, encoding="utf-8") as tmp:
    tmp.write(melody_clean)
    melody_tmp_path = pathlib.Path(tmp.name)

try:
    score = abc.load_abc(melody_tmp_path)
    display(HTML("<b>Soprano melody (V:1 only):</b>"))
    melody_audio, sr = abc.sonify_part(score, 0, sf2_path=SF2_PATH)
    display(abc.play_audio(melody_audio, sr))
finally:
    melody_tmp_path.unlink(missing_ok=True)

## 3. Run the Harmonization Pipeline

This cell runs all three agents:

1. **Theory Agent** (Claude Sonnet 4.6) analyzes the melody
2. **Harmonizer Agent** (GPT-4o) generates the chord progression
3. **Orchestrator Agent** (GPT-4o-mini) validates the final ABC

The `HarmonizationResult` contains all three pieces of data.

In [ ]:
# Run the full pipeline (async — use await in a Jupyter cell or asyncio.run() in a script)
print("Running 3-agent harmonization pipeline...")
print("  Step 1: Theory Agent analyzing melody...")
print("  Step 2: Harmonizer Agent generating chords...")
print("  Step 3: Orchestrator Agent cleaning up ABC...")
print()

result = await harmonize_melody(melody_clean)

print("Pipeline complete!")

## 4. Theory Agent Output

Here's the **harmonic analysis** produced by the Theory Agent (Claude Sonnet 4.6).
It shows the Roman numeral chord choices and functional labels for each measure.

In [ ]:
display(HTML("<h3>Theory Agent Analysis (Claude Sonnet 4.6)</h3>"))
display(HTML(f"<pre style='background:#f5f5f5; padding:12px; border-radius:6px'>{result.analysis}</pre>"))

## 5. Harmonizer Agent Output

Here's the **final 2-voice ABC** generated by the Harmonizer Agent (GPT-4o),
incorporating the theory analysis. V:2 now contains block chords instead of rests.

In [ ]:
display(HTML("<h3>Final 2-Voice ABC (GPT-4o + GPT-4o-mini cleanup)</h3>"))
display(HTML(f"<pre style='background:#f0f8ff; padding:12px; border-radius:6px'>{result.final_abc}</pre>"))

## 6. Listen to the Harmonized Result

Parse the final ABC with `util.load_abc()` and synthesize both voices together.

In [ ]:
with tempfile.NamedTemporaryFile(mode="w", suffix=".abc", delete=False, encoding="utf-8") as tmp:
    tmp.write(result.final_abc)
    result_tmp_path = pathlib.Path(tmp.name)

try:
    score = abc.load_abc(result_tmp_path)
    parts = abc.list_parts(score)

    # Show voice timing info (both voices should start at t=0)
    for p in parts:
        t0 = p.get("start_time", 0)
        color = "green" if t0 < 0.5 else "red"
        display(HTML(
            f'<small style="color:{color}">'
            f'[{p["instrument_index"]}] {p["name"]} — '
            f'start: {t0:.2f}s, end: {p.get("end_time", "?"):.2f}s'
            f'</small><br>'
        ))

    # Full mix: melody + chords
    display(HTML("<b>Melody (V:1) + Chords (V:2):</b>"))
    mix_audio, sr = abc.sonify_parts(score, [0, 1], sf2_path=SF2_PATH)
    display(abc.play_audio(mix_audio, sr))

    # Chords only (V:2)
    try:
        display(HTML("<b>Chords only (V:2):</b>"))
        chords_audio, sr = abc.sonify_part(score, 1, sf2_path=SF2_PATH)
        display(abc.play_audio(chords_audio, sr))
    except Exception as e:
        display(HTML(f'<p style="color:orange">Could not isolate chords: {e}</p>'))

    meta = abc.get_metadata(score)
    display(HTML(
        f"<small>Key: {meta['key']} | Tempo: {meta['tempo_bpm']} BPM | "
        f"Parts: {meta['num_parts']} | Duration: {meta['duration_seconds']:.1f}s</small>"
    ))

except RuntimeError as e:
    display(HTML(f'<p style="color:red"><b>abc2midi parse error:</b> {e}</p>'))
    display(HTML(
        "<details><summary>Raw ABC (click to expand)</summary>"
        f"<pre>{result.final_abc}</pre></details>"
    ))

finally:
    result_tmp_path.unlink(missing_ok=True)

## 7. Try a Different BWV

Now let's run the same pipeline on a different Bach chorale to see how the
agents adapt to a different key and melodic character.

In [ ]:
BWV_2 = "bwv269"   # Change to bwv255, bwv274, etc.

print(f"Loading {BWV_2.upper()}...")
melody_2 = load_bach_melody(BWV_2, measures=(1, 8))
template_2 = build_harmonization_template(melody_2, title_override=f"{BWV_2.upper()} Soprano")
clean_2 = clean_abc_for_llm(template_2)

print(f"Running pipeline on {BWV_2.upper()}...")
result_2 = await harmonize_melody(clean_2)

print("Done!")

In [ ]:
display(HTML(f"<h3>Theory Analysis for {BWV_2.upper()}</h3>"))
display(HTML(f"<pre style='background:#f5f5f5; padding:12px; border-radius:6px'>{result_2.analysis}</pre>"))

# Sonify the second result
with tempfile.NamedTemporaryFile(mode="w", suffix=".abc", delete=False, encoding="utf-8") as tmp:
    tmp.write(result_2.final_abc)
    result2_path = pathlib.Path(tmp.name)

try:
    score_2 = abc.load_abc(result2_path)
    display(HTML(f"<b>{BWV_2.upper()} — Melody + Chords:</b>"))
    mix_2, sr = abc.sonify_parts(score_2, [0, 1], sf2_path=SF2_PATH)
    display(abc.play_audio(mix_2, sr))
except RuntimeError as e:
    display(HTML(f'<p style="color:red">abc2midi parse error: {e}</p>'))
    display(HTML(f"<pre>{result_2.final_abc}</pre>"))
finally:
    result2_path.unlink(missing_ok=True)

## 8. Architecture Recap

Here's a summary of the 3-agent framework:

| Component | File | Role | Model |
|-----------|------|------|-------|
| `TheoryExecutor` | `executors.py` | Wraps Theory Agent, handles `MelodyInput → TheoryAnalysis` | Claude Sonnet 4.6 |
| `HarmonizerExecutor` | `executors.py` | Wraps Harmonizer Agent, handles `TheoryAnalysis → HarmonizationResult` | GPT-4o |
| `OrchestratorAgent` | `agents.py` | Post-process cleanup (not in workflow graph) | GPT-4o-mini |
| `WorkflowBuilder` | `pipeline.py` | Connects executors into a directed graph | — |
| `music_theory_context.py` | — | OMT chapter text injected into system prompts | — |
| `bach_melodies.py` | — | music21 corpus → ABC template utility | — |

### Message Types
```
MelodyInput(melody_abc: str)
    ↓  [TheoryExecutor]
TheoryAnalysis(melody_abc: str, analysis: str)
    ↓  [HarmonizerExecutor]
HarmonizationResult(melody_abc: str, analysis: str, final_abc: str)
```

### Changing Models
To swap models, edit `agents.py`:
```python
# Use a different Anthropic model for theory analysis
create_theory_agent(model="claude-opus-4-6")

# Use GPT-4o-mini for faster (cheaper) harmonization
create_harmonizer_agent(model="gpt-4o-mini")
```